In [1]:
# module to indicesise data (Jan 2020 as base 100)
import pandas as pd
import numpy as np


In [ ]:

def indexise(data, yoycolnum = 1, momcolnum = 2, base = "2020-01-01", percentage = True):
    data['Date'] = pd.to_datetime(data['Date'])
    data = data.sort_values('Date')
    
    data_yoy = data.iloc[:, [0, yoycolnum]].dropna()
    data_mom = data.iloc[:, [0, momcolnum]].dropna()
    date_end = data['Date'].max()
    date_yoy_start = data_yoy['Date'].min() - pd.DateOffset(months=12)
    date_mom_start = data_mom['Date'].min() - pd.DateOffset(months=1)

    # indice dataframe initialisation
    indice = pd.DataFrame(columns=['Date', 'Index Y', 'Index M', 'Index'])
    col_rec = len(indice.columns) - 2
    indice['Date'] = pd.date_range(start=min(date_yoy_start, date_mom_start), end=date_end, freq='MS')
    indice['Date'] = pd.to_datetime(indice['Date'])
    indice = indice.merge(data, on='Date', how='left')
    # after merging make sure yoycolnum and momcolnum still work
    indice = indice.set_index('Date')
    indice.loc[base, ["Index Y", "Index M", "Index"]] = 100

    # base year computation
    base_index = indice.index.get_loc(pd.Timestamp(base))
    base_next_index = base_index + 12
    for i in range(base_index + 1, base_next_index):
        # index = previous index cum mom
        indice.iloc[i, 2] = indice.iloc[i - 1, 2] * (1 + indice.iloc[i, momcolnum+col_rec] / 100)

    # computation of first data following base year
    # indice.iloc[base_next_index, 0] = indice.iloc[base_index, 2] * (1 + indice.iloc[base_next_index, yoycolnum+col_rec] / 100) # yoy
    # indice.iloc[base_next_index, 1] = indice.iloc[base_next_index - 1, 2] * (1 + indice.iloc[base_next_index, momcolnum+col_rec] / 100) # mom
    # indice.iloc[base_next_index, 2] = indice.iloc[base_next_index, [0, 1]].mean()

    # computation of following years
    for i in range(base_next_index, len(indice)):
        # index_y = previous year, same month index cum yoy
        if pd.isna(indice.iloc[i, yoycolnum+col_rec]):
            indice.iloc[i, 0] = pd.NA
        else:
            indice.iloc[i, 0] = indice.iloc[i - 12, 2] * (1 + indice.iloc[i, yoycolnum+col_rec] / 100)

        # index_m = previous month index cum mom
        if pd.isna(indice.iloc[i, momcolnum+col_rec]):
            indice.iloc[i, 1] = pd.NA
        else:
            indice.iloc[i, 1] = indice.iloc[i - 1, 2] * (1 + indice.iloc[i, momcolnum+col_rec] / 100)

        # index = average of index_y and index_m
        indice.iloc[i, 2] = indice.iloc[base_next_index, [0, 1]].mean()

    return indice.loc['Index']
    

    


In [2]:
data = pd.read_csv('CN PPI Output General (%).csv', date_format='%Y-%m-%d')
data = data[['Date', 'PPI Output - MoS (Food) YoY', 'PPI Output - MoS (Food) MoM']]
data['Date'] = pd.to_datetime(data['Date'])
data = data.sort_values('Date', ascending=True)

data_yoy = data.iloc[:, [0, 1]].dropna()
data_mom = data.iloc[:, [0, 2]].dropna()
date_end = data['Date'].max()
date_yoy_start = data_yoy['Date'].min() - pd.DateOffset(months=12)
date_mom_start = data_mom['Date'].min() - pd.DateOffset(months=1)

indice = pd.DataFrame(columns=['Date', 'Index Y', 'Index M', 'Index'])
indice['Date'] = pd.date_range(start=min(date_yoy_start, date_mom_start), end=date_end, freq='MS')
indice['Date'] = pd.to_datetime(indice['Date'])
indice = indice.merge(data, on='Date', how='left')

indice = indice.set_index('Date')
indice.loc["2020-01-01", ["Index Y", "Index M", "Index"]] = 100

In [7]:
data.head()

,Date,PPI Output - MoS (Food) YoY,PPI Output - MoS (Food) MoM
399,1993-01-01,NaN,NaN
398,1993-02-01,NaN,NaN
397,1993-03-01,NaN,NaN
396,1993-04-01,NaN,NaN
395,1993-05-01,NaN,NaN


In [3]:
indice.head()

,Index Y,Index M,Index,PPI Output - MoS (Food) YoY,PPI Output - MoS (Food) MoM
Date,,,,,
1995-10-01,NaN,NaN,NaN,NaN,NaN
1995-11-01,NaN,NaN,NaN,NaN,NaN
1995-12-01,NaN,NaN,NaN,NaN,NaN
1996-01-01,NaN,NaN,NaN,NaN,NaN
1996-02-01,NaN,NaN,NaN,NaN,NaN


In [6]:
len(indice.columns)

5

In [8]:
indice.iloc[291,0]

100

In [240]:
base = "2020-01-01"
indice.index.get_loc(pd.Timestamp(base))

291

In [244]:
indice_2020 = indice[indice.index.year == 2020]
for i in range(len(indice_2020)-1):
    # index = previous index cum mom
    indice_2020.iloc[i+1, 2] = indice_2020.iloc[i, 2] * (1 + indice_2020.iloc[i, 4]/100)

indice.loc[indice.index.year == 2020, "Index"] = indice_2020['Index']

In [179]:
indice.index.get_loc(pd.Timestamp("2021-01-01"))

303

In [246]:
start_index_2021 = indice.index.get_loc(pd.Timestamp("2021-01-01"))

indice.iloc[start_index_2021,0] = indice.iloc[indice.index.get_loc(pd.Timestamp("2020-01-01")),2] * (1 + indice.iloc[indice.index.get_loc(pd.Timestamp("2021-01-01")),-2] / 100)
indice.iloc[start_index_2021,1] = indice.iloc[start_index_2021-1,2] * (1 + indice.iloc[start_index_2021,-1] / 100)
indice.iloc[start_index_2021,2] = (indice.iloc[start_index_2021,0] + indice.iloc[start_index_2021,1])/2

indice.loc[indice.index.year >= 2020].head(13)

,Index Y,Index M,Index,PPI Output - MoS (Food) YoY,PPI Output - MoS (Food) MoM
Date,,,,,
2020-01-01,100,100,100,5.1,0.0
2020-02-01,NaN,NaN,100.0,5.1,0.0
2020-03-01,NaN,NaN,100.0,4.5,-0.3
2020-04-01,NaN,NaN,99.7,3.7,-0.1
2020-05-01,NaN,NaN,99.6003,2.9,-0.5
2020-06-01,NaN,NaN,99.102299,3.2,0.3
2020-07-01,NaN,NaN,99.399605,3.7,0.6
2020-08-01,NaN,NaN,99.996003,3.1,0.2
2020-09-01,NaN,NaN,100.195995,2.1,0.0


In [ ]:
# compute the index from 2021 on

for i in range(start_index_2021 + 1, len(indice)):
    # index_y = previous year, same month index cum yoy
    indice.iloc[i, 0] = indice.iloc[i - 12, 2] * (1 + indice.iloc[i, -2] / 100)
    # index_m = previous month index cum mom
    indice.iloc[i, 1] = indice.iloc[i - 1, 2] * (1 + indice.iloc[i, -1] / 100)
    # index = average of index_y and index_m
    indice.iloc[i, 2] = (indice.iloc[i, 0] + indice.iloc[i, 1]) / 2

indice[indice.index.year>=2021]

,Index Y,Index M,Index,PPI Output - MoS (Food) YoY,PPI Output - MoS (Food) MoM
Date,,,,,
2021-01-01,105.1,100.797068,102.948534,1.6,0.7
2021-02-01,101.6,103.154431,102.377215,1.6,0.2
2021-03-01,102.0,102.58197,102.290985,2.0,0.2
2021-04-01,101.4946,101.984112,101.739356,1.8,-0.3
2021-05-01,101.791507,101.739356,101.765431,2.2,0.0
...,...,...,...,...,...
2025-12-01,100.749553,100.893366,100.82146,-1.5,-0.1
2026-01-01,100.247913,100.720638,100.484275,-1.9,-0.1
2026-02-01,100.323163,100.484275,100.403719,-1.8,0.0


In [228]:
# compute the index till 2019 on
for i in range(start_index_2021-13, -1, -1):
    # index_y = previous year, same month index cum yoy
    if pd.isna(indice.iloc[i + 12, -2]):
        indice.iloc[i, 0] = pd.NA
    else: 
        indice.iloc[i, 0] = indice.iloc[i + 12, 2] / (1 + indice.iloc[i + 12, -2] / 100)
    # index_m = previous month index cum mom
    if pd.isna(indice.iloc[i + 1, -1]):
        indice.iloc[i, 1] = pd.NA
    else: 
        indice.iloc[i, 1] = indice.iloc[i + 1, 2] / (1 + indice.iloc[i + 1, -1] / 100)
    # index = average of index_y and index_m
    indice.iloc[i, 2] = indice.iloc[i, [0, 1]].mean()


In [232]:
indice[indice.index.year<=2010]

,Index Y,Index M,Index,PPI Output - MoS (Food) YoY,PPI Output - MoS (Food) MoM
Date,,,,,
1995-10-01,69.403865,<NA>,69.403865,NaN,NaN
1995-11-01,70.846643,<NA>,70.846643,NaN,NaN
1995-12-01,71.473791,<NA>,71.473791,NaN,NaN
1996-01-01,71.27824,<NA>,71.27824,NaN,NaN
1996-02-01,71.077533,<NA>,71.077533,NaN,NaN
...,...,...,...,...,...
2010-08-01,84.67948,<NA>,84.67948,4.0,NaN
2010-09-01,85.250267,<NA>,85.250267,4.7,NaN
2010-10-01,85.852873,<NA>,85.852873,5.3,NaN
